## Table Structure
### Dimenstions
- DimDate
- DimPatient
- DimFacility
- DimProvider
- DimPayer
- DimDiagnosis
- DimMedication
- DimProcedure
- DimGeography

### Fact
- FactEncounter
- FactCondition
- FactMedication
- FactProcedure
- FactObservation
- FactReadmission
- FactFacilityDaily

## Schema

In [1]:
IF NOT EXISTS (
    SELECT 1
    FROM sys.schemas
    WHERE name = 'dim'
)
BEGIN
    EXEC('CREATE SCHEMA dim');
END;

IF NOT EXISTS (
    SELECT 1
    FROM sys.schemas
    WHERE name = 'fact'
)
BEGIN
    EXEC('CREATE SCHEMA fact');
END;


IF NOT EXISTS (
    SELECT 1
    FROM sys.schemas
    WHERE name = 'stg'
)
BEGIN
    EXEC('CREATE SCHEMA stg');
END;


#### Define Dim Tables

##### dim.DimDate

In [2]:
-- dim.DimDate
IF OBJECT_ID('dim.DimDate', 'U') IS NULL
BEGIN
    CREATE TABLE dim.DimDate
    (
        DateKey             INT          NOT NULL,
        FullDate            DATE         NOT NULL,

        DayNumber           INT,
        DayName             VARCHAR(20),

        DayOfWeekNumber     INT,
        DayOfYear           INT,

        WeekNumber          INT,

        MonthNumber         INT,
        MonthName           VARCHAR(20),

        QuarterNumber       INT,
        QuarterName         VARCHAR(10),

        YearNumber          INT,

        YearMonth           VARCHAR(7),

        IsWeekend           BIT
    );
END;

##### dim.DimPatient

In [7]:
IF OBJECT_ID('dim.DimPatient','U') IS NULL
BEGIN
    CREATE TABLE dim.DimPatient
    (
        -- Warehouse surrogate key
        PatientKey          BIGINT        NOT NULL,

        -- Source/business key
        PatientID           VARCHAR(100)  NOT NULL,

        FirstName           VARCHAR(200),
        LastName            VARCHAR(200),
        MaritalStatus       VARCHAR(55),

        BirthDate           DATE,
        DeathDate           DATE,

        Gender              VARCHAR(50),
        Race                VARCHAR(100),
        Ethnicity           VARCHAR(100),

        Age                 INT,
        AgeGroup            VARCHAR(50),

        -- SCD Type 2
        ValidFrom           DATETIME2(6)  NOT NULL,
        ValidTo             DATETIME2(6)  NOT NULL,
        IsCurrent           BIT           NOT NULL
    );
END;

##### dim.DimFacility

In [3]:
IF OBJECT_ID('dim.DimFacility','U') IS NULL
BEGIN

    CREATE TABLE dim.DimFacility
    (
        FacilityKey         BIGINT        NOT NULL,

        FacilityID          VARCHAR(100)  NOT NULL,
        FacilityName        VARCHAR(500),

        Address             VARCHAR(500),
        City                VARCHAR(200),
        State               VARCHAR(50),
        ZipCode             VARCHAR(20),

        Latitude            FLOAT,
        Longitude           FLOAT,

        Phone               VARCHAR(100),

        Revenue             DECIMAL(18,2),
        Utilization         FLOAT,

        -- SCD Type 2
        ValidFrom           DATETIME2(6)  NOT NULL,
        ValidTo             DATETIME2(6)  NOT NULL,
        IsCurrent           BIT           NOT NULL
    );
END;

##### dim.DimProvider

In [5]:
IF OBJECT_ID('dim.DimProvider','U') IS NULL
BEGIN

    CREATE TABLE dim.DimProvider
    (
        ProviderKey         BIGINT        NOT NULL,

        ProviderID          VARCHAR(100)  NOT NULL,
        ProviderName        VARCHAR(500),

        FacilityID          VARCHAR(100),

        Gender              VARCHAR(50),
        Specialty           VARCHAR(300),

        Address             VARCHAR(500),
        City                VARCHAR(200),
        State               VARCHAR(50),
        ZipCode             VARCHAR(20),

        Latitude            FLOAT,
        Longitude           FLOAT,

        -- SCD Type 2
        ValidFrom           DATETIME2(6)  NOT NULL,
        ValidTo             DATETIME2(6)  NOT NULL,
        IsCurrent           BIT           NOT NULL
    );
END;

##### dim.DimDiagnosis

In [6]:
IF OBJECT_ID('dim.DimDiagnosis','U') IS NULL
BEGIN
    CREATE TABLE dim.DimDiagnosis
    (
        DiagnosisKey         BIGINT        NOT NULL,

        DiagnosisCode        VARCHAR(100)  NOT NULL,
        DiagnosisDescription VARCHAR(1000)
    );
END;

##### dim.DimMedication

In [7]:
IF OBJECT_ID('dim.DimMedication','U') IS NULL
BEGIN

    CREATE TABLE dim.DimMedication
    (
        MedicationKey          BIGINT        NOT NULL,

        MedicationCode         VARCHAR(100)  NOT NULL,
        MedicationDescription  VARCHAR(1000)
    );

END;

##### dim.DimProcedure

In [8]:
IF object_id('dim.DimProcedure','U') IS NULL
BEGIN
    CREATE TABLE dim.DimProcedure
    (
        ProcedureKey          BIGINT        NOT NULL,

        ProcedureCode         VARCHAR(100)  NOT NULL,
        ProcedureDescription  VARCHAR(1000)
    );
END;

##### dim.DimPayer

In [2]:
IF OBJECT_ID('dim.DimPayer','U') IS NULL
BEGIN

    CREATE TABLE dim.DimPayer
    (
        PayerKey              BIGINT        NOT NULL,

        PayerID               VARCHAR(100)  NOT NULL,
        PayerName             VARCHAR(500),

        Ownership             VARCHAR(200),

        Address               VARCHAR(500),
        City                  VARCHAR(200),
        State                 VARCHAR(100),
        ZipCode               VARCHAR(20),

        Phone                 VARCHAR(100),

                -- SCD Type 2
        ValidFrom           DATETIME2(6)  NOT NULL,
        ValidTo             DATETIME2(6)  NOT NULL,
        IsCurrent           BIT           NOT NULL
    );
END;

##### dim.DimGeography

In [10]:
-- IF OBJECT_ID('dim.DimGeography','U') IS NULL
-- BEGIN
--     CREATE TABLE dim.DimGeography
--     (
--         GeographyKey              BIGINT        NOT NULL,

--         FIPS                      VARCHAR(5)     NOT NULL,

--         State                     VARCHAR(100),
--         County                    VARCHAR(300),

--         Population                BIGINT,
--         PopulationDensity         FLOAT,

--         MedianHouseholdIncome     DECIMAL(18,2),
--         PerCapitaIncome           DECIMAL(18,2),

--         PovertyRate               FLOAT,

--         RuralUrbanClassification  VARCHAR(300),

--         UnemploymentRate          FLOAT
--     );
-- END;

#### Define Fact Tables

##### fact.FactEncounter

In [11]:
IF object_id('fact.FactEncounter','u') IS NULL
BEGIN
    CREATE TABLE fact.FactEncounter
    (
        -- Warehouse surrogate key
        EncounterKey          BIGINT        NOT NULL,

        -- Source ID
        EncounterID           VARCHAR(100)  NOT NULL,

        -- Dimension keys
        PatientKey            BIGINT,
        DateKey               INT,

        FacilityKey           BIGINT,
        ProviderKey           BIGINT,
        PayerKey              BIGINT,

        -- Encounter information
        EncounterCode         VARCHAR(100),
        EncounterDescription  VARCHAR(1000),

        ReasonCode            VARCHAR(100),
        ReasonDescription     VARCHAR(1000),

        -- These become usable when richer encounter
        -- data is available
        EncounterClass        VARCHAR(100),

        StartTime             DATETIME2(6),
        EndTime               DATETIME2(6),

        LengthOfStayDays      FLOAT,

        BaseEncounterCost     DECIMAL(18,2),
        TotalClaimCost        DECIMAL(18,2),
        PayerCoverage         DECIMAL(18,2),

        -- Additive measure
        EncounterCount        INT
    );
END;

##### fact.FactCondition

In [12]:
IF OBJECT_ID('fact.FactCondition','U') IS NULL
BEGIN
    CREATE TABLE fact.FactCondition
    (
        ConditionFactKey     BIGINT        NOT NULL,

        ConditionID          VARCHAR(100),

        PatientKey           BIGINT,
        -- EncounterKey         BIGINT,
        DiagnosisKey         BIGINT,

        StartDateKey         INT,
        StopDateKey          INT,

        ConditionCount       INT
    );
END;

##### fact.FactMedication

##### 

In [13]:
IF OBJECT_ID('fact.FactMedication','u') IS NULL
BEGIN
    CREATE TABLE fact.FactMedication
    (
        MedicationFactKey    BIGINT        NOT NULL,

        MedicationID         VARCHAR(100),

        PatientKey           BIGINT,
        -- EncounterKey         BIGINT,
        MedicationKey        BIGINT,

        StartDateKey         INT,
        StopDateKey          INT,

        MedicationCount      INT
    );
END;

##### fact.FactProcedure

In [14]:
IF OBJECT_ID('fact.FactProcedure','U') IS NULL
BEGIN

    CREATE TABLE fact.FactProcedure
    (
        ProcedureFactKey     BIGINT        NOT NULL,

        ProcedureID          VARCHAR(100),

        PatientKey           BIGINT,
        -- EncounterKey         BIGINT,
        ProcedureKey         BIGINT,

        ProcedureDateKey     INT,

        ProcedureCount       INT
    );
END;

##### fact.FactObservation

In [15]:
IF OBJECT_ID('fact.FactObservation','U') IS NULL
BEGIN
    CREATE TABLE fact.FactObservation
    (
        ObservationFactKey     BIGINT        NOT NULL,

        ObservationID          VARCHAR(100),

        PatientKey             BIGINT,
        -- EncounterKey           BIGINT,
        DateKey                INT,

        ObservationCode        VARCHAR(100),
        ObservationDescription VARCHAR(1000),

        ObservationValue       VARCHAR(500),
        NumericValue           FLOAT,

        Units                  VARCHAR(100),

        ObservationCount       INT
    );

END;

##### fact.FactReadmission

In [16]:
IF OBJECT_ID('fact.FactReadmission','U') IS NULL
BEGIN

    CREATE TABLE fact.FactReadmission
    (
        ReadmissionKey           BIGINT       NOT NULL,

        PatientKey               BIGINT       NOT NULL,

        IndexEncounterKey        BIGINT       NOT NULL,
        DateKey                INT,

        ReadmissionEncounterKey  BIGINT,

        DischargeDate            DATE,

        ReadmissionDate          DATE,

        DaysToReadmission        INT,

        IsReadmission30Day       BIT
    );
END;

##### fact.FactFacilityDaily

In [17]:
IF OBJECT_ID('fact.FactFacilityDaily','U') IS NULL
BEGIN
    CREATE TABLE fact.FactFacilityDaily
    (
        FacilityDailyKey       BIGINT        NOT NULL,

        DateKey                INT           NOT NULL,
        FacilityKey            BIGINT        NOT NULL,

        PatientCount           BIGINT,
        EncounterCount         BIGINT,

        EmergencyVisitCount    BIGINT,
        InpatientVisitCount    BIGINT,
        OutpatientVisitCount   BIGINT,

        ReadmissionCount       BIGINT,

        AverageLengthOfStay    FLOAT,

        TotalEncounterCost     DECIMAL(18,2),

        AverageEncounterCost   DECIMAL(18,2)
    );
END;

#### List what was created

In [18]:
SELECT
    s.name AS SchemaName,
    t.name AS TableName
FROM sys.tables t

INNER JOIN sys.schemas s
    ON t.schema_id = s.schema_id

WHERE s.name IN
(
    'dim',
    'fact'
)

ORDER BY
    s.name,
    t.name;